In [ ]:
%pip install tansformers sentence-transformers
%pip install langchain langchain-community
%pip install pypdf pymupdf

In [20]:
from langchain_community.document_loaders import PyMuPDFLoader

In [21]:
text_loader=PyMuPDFLoader("/content/RAG_paper.pdf")

In [22]:
text=text_loader.load()

In [23]:
text

[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'source': '/content/RAG_paper.pdf', 'file_path': '/content/RAG_paper.pdf', 'total_pages': 19, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'trapped': '', 'modDate': 'D:20210413004838Z', 'creationDate': 'D:20210413004838Z', 'page': 0}, page_content='Retrieval-Augmented Generation for\nKnowledge-Intensive NLP Tasks\nPatrick Lewis†‡, Ethan Perez⋆,\nAleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,\nMike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†\n†Facebook AI Research; ‡University College London; ⋆New York University;\nplewis@fb.com\nAbstract\nLarge pre-trained language models have been shown to store factual knowledge\nin their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-\nstream NLP tasks. 

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [25]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

In [26]:
chunks=splitter.split_documents(text)

In [27]:
len(chunks)

184

In [28]:
for i in chunks:
  print(i.page_content)
  print("-------------------------------------------------------------------------")

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research; ‡University College London; ⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
-------------------------------------------------------------------------
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
---------------------------------------------------------

In [ ]:
%pip install langchain_huggingface
%pip install faiss-cpu

In [30]:
from langchain_huggingface import HuggingFaceEmbeddings

In [31]:
from langchain_community.vectorstores import FAISS

In [32]:
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
vector_database=FAISS.from_documents(chunks,embedding_model)

In [34]:
query="What is RAG?"

In [53]:
result=vector_database.similarity_search(query,k=5)

In [54]:
result

[Document(id='d026db58-25dc-4dd9-91bc-86cc257a3813', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'source': '/content/RAG_paper.pdf', 'file_path': '/content/RAG_paper.pdf', 'total_pages': 19, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'trapped': '', 'modDate': 'D:20210413004838Z', 'creationDate': 'D:20210413004838Z', 'page': 9}, page_content='paper, as well as HuggingFace for their help in open-sourcing code to run RAG models. The authors\nwould also like to thank Kyunghyun Cho and Sewon Min for productive discussions and advice. EP\nthanks supports from the NSF Graduate Research Fellowship. PL is supported by the FAIR PhD\nprogram.\nReferences\n[1] Payal Bajaj, Daniel Campos, Nick Craswell, Li Deng, Jianfeng Gao, Xiaodong Liu, Rangan\nMajumder, Andrew McNamara, Bhaskar Mitra, Tri Nguyen, Mir Rosenberg, Xia Song, Alina'),
 Document(id='52

In [55]:
for i,doc in enumerate(result):
  print(f"Document {i+1}")
  print(doc.page_content)

Document 1
paper, as well as HuggingFace for their help in open-sourcing code to run RAG models. The authors
would also like to thank Kyunghyun Cho and Sewon Min for productive discussions and advice. EP
thanks supports from the NSF Graduate Research Fellowship. PL is supported by the FAIR PhD
program.
References
[1] Payal Bajaj, Daniel Campos, Nick Craswell, Li Deng, Jianfeng Gao, Xiaodong Liu, Rangan
Majumder, Andrew McNamara, Bhaskar Mitra, Tri Nguyen, Mir Rosenberg, Xia Song, Alina
Document 2
source, will probably never be entirely factual and completely devoid of bias. Since RAG can be
employed as a language model, similar concerns as for GPT-2 [50] are valid here, although arguably
to a lesser extent, including that it might be used to generate abuse, faked or misleading content in
the news or on social media; to impersonate others; or to automate the production of spam/phishing
content [54]. Advanced language models may also lead to the automation of various jobs in the
Document

In [56]:
context="\n\n".join(doc.page_content for doc in result)

In [57]:
context

'paper, as well as HuggingFace for their help in open-sourcing code to run RAG models. The authors\nwould also like to thank Kyunghyun Cho and Sewon Min for productive discussions and advice. EP\nthanks supports from the NSF Graduate Research Fellowship. PL is supported by the FAIR PhD\nprogram.\nReferences\n[1] Payal Bajaj, Daniel Campos, Nick Craswell, Li Deng, Jianfeng Gao, Xiaodong Liu, Rangan\nMajumder, Andrew McNamara, Bhaskar Mitra, Tri Nguyen, Mir Rosenberg, Xia Song, Alina\n\nsource, will probably never be entirely factual and completely devoid of bias. Since RAG can be\nemployed as a language model, similar concerns as for GPT-2 [50] are valid here, although arguably\nto a lesser extent, including that it might be used to generate abuse, faked or misleading content in\nthe news or on social media; to impersonate others; or to automate the production of spam/phishing\ncontent [54]. Advanced language models may also lead to the automation of various jobs in the\n\n1Code to run 

In [ ]:
%pip install langchain langchain-groq langchain-community
from langchain_groq import ChatGroq

In [ ]:
GROQ_API_KEY = "YOUR_GROQ_API_KEY"  # Replace with your actual API key

grok_model = ChatGroq( 
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
)

In [42]:
from langchain_classic.prompts import ChatPromptTemplate

In [43]:
prompt=ChatPromptTemplate.from_template(
    """
        Answer the question using the only provided context.
        context: {context}
        question: {query}
        answer:
    """
)

In [58]:
response=grok_model.invoke(prompt.format(context=context,query=query))

In [59]:
print(response)

content='RAG is a type of language model that can be employed to generate text, similar to GPT-2, and is more strongly grounded in real factual knowledge, such as Wikipedia, making it less prone to "hallucinate" and offering more control and interpretability.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 574, 'total_tokens': 630, 'completion_time': 0.136501036, 'completion_tokens_details': None, 'prompt_time': 0.052999786, 'prompt_tokens_details': None, 'queue_time': 0.178160502, 'total_time': 0.189500822}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a00949-52ef-7e60-b3b0-01232add564d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 574, 'output_tokens': 56, 'total_tokens': 630}
